# Lab 1 — Trace an Internal-Operations Assistant

**Required · 45 minutes**

## What will you do?

You will ask an AI assistant one practice question. Then you will use Azure AI Foundry to see what happened while the answer was created.

This information is called **telemetry**. Think of it as an activity log for your application.

- A **trace** is the full journey of one question.
- A **span** is one step in that journey.
- An **attribute** is a label that helps you find or understand a step.

Tracing is useful when an AI application is slow, gives an unexpected answer, or fails. It helps you see where the request went and how long each step took.

In this lab you will:

1. Load the workshop settings.
2. Turn on tracing.
3. Create a small AI assistant.
4. Ask it one practice question.
5. Find the request in Azure.

The basic flow is:

```text
Your question → AI assistant → model → answer
```


## 1. Load the workshop settings

This cell gets the information needed to connect to your Azure AI Foundry project and model.

It also creates a **namespace**. A namespace is simply a short label for you or your team. It keeps your work separate from the work of other participants.

You should not need to change this code. The workshop setup should already have created the required settings in a `.env` file.

**Run the cell. You should see:**

- where the settings were found;
- your namespace;
- the installed version of `azure-ai-projects`.

If you see `ValueError`, one or more workshop settings are missing. Check your `.env` file or ask the facilitator.

In [ ]:
import os
import re
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv

def load_repo_env() -> Path | None:
    start = Path.cwd().resolve()
    for folder in (start, *start.parents):
        candidate = folder / '.env'
        if candidate.exists():
            load_dotenv(candidate)
            return candidate
    return None

env_path = load_repo_env()
endpoint = os.getenv('FOUNDRY_PROJECT_ENDPOINT') or os.getenv('AZURE_AI_PROJECT_ENDPOINT')
model_deployment = os.getenv('FOUNDRY_MODEL') or os.getenv('AZURE_AI_MODEL_DEPLOYMENT_NAME')
team_id = os.getenv('WORKSHOP_TEAM_ID', '').strip()
participant_id = os.getenv('WORKSHOP_PARTICIPANT_ID', '').strip()
configured_namespace = os.getenv('WORKSHOP_RESOURCE_NAMESPACE', '').strip()
raw_namespace = configured_namespace or team_id or participant_id
resource_namespace = re.sub(r'[^a-z0-9-]+', '-', raw_namespace.lower()).strip('-')[:32]
if not endpoint or not model_deployment or not resource_namespace:
    raise ValueError('Set the Foundry endpoint, model deployment, and workshop namespace')

def workshop_name(prefix: str) -> str:
    return f'{prefix}-{resource_namespace}'

print({'env': str(env_path) if env_path else 'process environment', 'namespace': resource_namespace, 'team': team_id, 'participant': participant_id})
print('azure-ai-projects', version('azure-ai-projects'))

## 2. Enable tracing and export it to Azure Monitor

This cell turns on the activity log for the AI request.

Two Azure services work together here:

- **Azure AI Foundry** runs the AI assistant.
- **Application Insights** stores and displays the trace.

The code uses **OpenTelemetry** to send the trace from the notebook to Application Insights. You do not need to understand its setup details for this lab.



**Run the cell. You should see:** `Tracing configured; sensitive message capture is disabled.`

In [ ]:
os.environ['AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING'] = 'true'
os.environ['OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT'] = 'false'
os.environ['AZURE_TRACING_GEN_AI_ENABLE_TRACE_CONTEXT_PROPAGATION'] = 'true'

from azure.identity import InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

credential = InteractiveBrowserCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
connection_string = project_client.telemetry.get_application_insights_connection_string()
if not connection_string:
    raise RuntimeError('Connect Application Insights to the Foundry project before this lab')

configure_azure_monitor(connection_string=connection_string)
AIProjectInstrumentor().instrument(
    enable_content_recording=False,
    enable_trace_context_propagation=True,
    enable_baggage_propagation=False,
)
tracer = trace.get_tracer('workshop.internal_operations')
# Create the OpenAI client only after instrumentation so trace context propagates.
openai_client = project_client.get_openai_client()
print('Tracing configured; sensitive message capture is disabled.')


## 3. Create the AI assistant

This cell creates a small AI assistant for the exercise.

A **model** creates text. An **agent** is a model with instructions about what it should do. This agent receives a few made-up facts from procedure `P-17`.

The instructions tell the agent to:

- use only the facts provided;
- mention procedure `P-17`;
- say what still needs to be checked;
- never claim that a real action was completed.

The cell also starts an empty conversation for the question you will ask next.

**Run the cell. You should see:** the assistant's name, version, and conversation ID.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = workshop_name('d2-observe')
agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            'You are a synthetic internal-operations assistant. '
            'Use only these workshop facts: P-17 requires isolation, absence-of-voltage verification, '
            'and a switching-authority confirmation before dispatch. '
            'Never claim that an operational action was executed. Cite procedure P-17. '
            'If required information is absent, state what must be verified.'
        ),
    ),
)
conversation = openai_client.conversations.create()
print({'agent': agent.name, 'version': agent.version, 'conversation': conversation.id})

## 4. Ask a question and record the journey

This cell asks the assistant an incident question.

At the same time, it creates a span called `workshop.incident_assistance`. This span records how long the request takes and adds a few simple labels, such as the team and scenario.

The code then creates a **trace ID**. Think of this as a tracking number for the request. You will use it to find the request in Azure.

**Run the cell. You should see:** the assistant's answer and a 32-character trace ID. The answer may take several seconds.

In [ ]:
synthetic_incident_id = 'SIM-1042'
with tracer.start_as_current_span('workshop.incident_assistance') as span:
    span.set_attribute('workshop.namespace', resource_namespace)
    span.set_attribute('workshop.team_id', team_id)
    span.set_attribute('workshop.scenario', 'procedure-guidance')
    span.set_attribute('incident.synthetic_id', synthetic_incident_id)
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            'agent_reference': {
                'name': agent.name,
                'id': agent.id,
                'type': 'agent_reference',
            }
        },
        input='For synthetic incident SIM-1042, what must be verified before dispatch?',
    )
    span.set_attribute('workshop.outcome', 'response-created')
    trace_id = f'{span.get_span_context().trace_id:032x}'

print(response.output_text)
print('Trace ID:', trace_id)

## 5. Check the result

This cell runs three small checks:

- a trace ID was created;
- the answer mentions `P-17`;
- the answer mentions something that must be checked or confirmed.

Python uses `assert` for these checks. If a check is false, the cell stops and shows an error.

**Run the cell. You should see:** `PASS — response contract and local trace ID checks succeeded.`

In [ ]:
answer = response.output_text.lower()
assert len(trace_id) == 32 and int(trace_id, 16) > 0
assert 'p-17' in answer, 'The synthetic answer must cite P-17'
assert any(term in answer for term in ('verify', 'verification', 'confirmation'))
print('PASS — response contract and local trace ID checks succeeded.')

## Challenge: record a smaller step

A request can contain several smaller steps. For example, answering the question may include looking up a procedure.

The main step is called the **parent span**. A smaller step inside it is called a **child span**.

In the next cell:

1. Start a new parent span.
2. Inside it, start a child span named `workshop.procedure_lookup`.
3. Look up `P-17` in the `PROCEDURES` dictionary.
4. Add only these labels: `procedure.id`, `lookup.hit`, and `workshop.namespace`.
5. Check that the parent and child have the same trace ID.

The child must be inside the parent's indented `with` block. That is how Python knows the two steps belong to the same request.


In [ ]:
# TODO: implement the privacy-safe child span.
PROCEDURES = {'P-17': {'requires_switching_authority': True}}

# with tracer.start_as_current_span('workshop.procedure_lookup') as lookup_span:
#     ...


## Find the trace in Azure

Now use the trace ID to find the request:

1. Open your project in Azure AI Foundry.
2. Go to Build -> Agents ->select your created agent -> Go to Traces -> Response View and see the response.
3. The traces and Conversation tabs will also show the conversation just with a small propagation delay, given they are coming from Application Insights.

It can take a couple of minutes for a new trace to appear.

Trace        = How was it processed?
Conversation = What happened across the chat?
Response     = What did the AI generate?

## Optional: explore further

Finished early? You can explore how to add the workshop namespace to every span automatically, or try Foundry trace evaluation.

Add a `SpanProcessor` that injects the workshop namespace into every span, or evaluate recent Application Insights traces with Foundry trace evaluation. Trace evaluation is preview; the deterministic inline datasets in the next labs are the fallback.

## Cleanup (optional)

This cell can delete the conversation and assistant created by this lab.

Cleanup is off by default so you can still view your trace. Run this cell after saving your trace URL or screenshot.

To remove the lab resources, set `WORKSHOP_ALLOW_CLEANUP=true` and run the cell. The code checks your namespace before deleting anything.

**You should see:** either a message that cleanup is disabled or a message confirming the deletion.

In [ ]:
allow_cleanup = os.getenv('WORKSHOP_ALLOW_CLEANUP', 'false').lower() == 'true'
expected_suffix = f'-{resource_namespace}'
if allow_cleanup:
    if not agent.name.endswith(expected_suffix):
        raise RuntimeError(f'Refusing to delete non-owned agent: {agent.name}')
    openai_client.conversations.delete(conversation_id=conversation.id)
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    print('Deleted this lab conversation and namespaced agent version.')
else:
    print('Cleanup disabled. Set WORKSHOP_ALLOW_CLEANUP=true to remove only this namespaced agent.')